In [1]:
import json
import pandas as pd
import os
from src.resume_matcher import ResumeJobMatcher
import sys
import os
from utils.deepseek_helper import DeepSeekHelper  # 导入AI分析助手
from datetime import datetime
ROOT_DIR = os.getcwd()
sys.path.append(ROOT_DIR)

In [6]:
def read_data():
    # 设置文件路径
    resume_path = os.path.join(ROOT_DIR, "data/generated_resumes_with_ids.json")
    job_path = os.path.join(ROOT_DIR, "data/SZTU_2025_SPRING_POSITION_TABLE.xlsx")
    # 读取简历数据
    try:
        with open(resume_path, 'r', encoding='utf-8') as f:
            resumes = json.load(f)
        print(f"成功读取到 {len(resumes)} 份简历信息")
        
        # 打印第一份简历作为示例
        print("\n示例简历信息:")
        first_resume = resumes[0]
        print(f"姓名: {first_resume['基本信息'].get('姓名', 'N/A')}")
        print(f"意向职位: {first_resume['意向岗位'].get('职位', 'N/A')}")
        print(f"教育背景: {first_resume['教育经历'].get('专业', 'N/A')} - {first_resume['教育经历'].get('学历', 'N/A')}")
    except Exception as e:
        print(f"读取简历数据失败: {e}")
        resumes = None

    # 读取企业数据
    try:
        jobs = pd.read_excel(job_path)
        print(f"\n成功读取到 {len(jobs)} 条企业岗位信息")
        
        # 打印前3条企业信息
        print("\n企业岗位信息示例(前3条):")
        #print(jobs.head(3)[['单位名称', '岗位名称', '最低薪资（K/月）', '最高薪资（K/月）']])
    except Exception as e:
        print(f"读取企业数据失败: {e}")
        jobs = None

    return resumes, jobs

In [7]:
resumes, jobs = read_data()
print(resumes[0])
print(jobs.head(3))

成功读取到 35 份简历信息

示例简历信息:
姓名: 王丽
意向职位: 数据分析师
教育背景: 金融学 - 本科

成功读取到 858 条企业岗位信息

企业岗位信息示例(前3条):
{'基本信息': {'姓名': '王丽', '性别': '女', '年龄': '22', '电话': '13812345678', '邮箱': 'wangli123@qq.com', '籍贯': '江苏省南京市', '唯一标识': '0001'}, '意向岗位': {'类型': '校招', '城市': '上海', '职位': '数据分析师', '行业': '金融业', '薪资': '12k-15k'}, '教育经历': {'专业': '金融学', '学历': '本科'}, '工作经历': '在知名金融公司实习期间，参与了多个数据分析项目，包括市场趋势分析、客户行为分析等，使用Python和R语言进行数据处理和建模。', '专业技能': '精通Python和R语言，熟悉SQL数据库操作，掌握数据可视化工具如Tableau和Power BI，具备良好的统计学基础和数据挖掘能力。', '项目经历': '主导了一个关于股市预测的项目，使用机器学习算法对历史数据进行分析，成功预测了股市的短期走势。', '校内荣誉': '连续三年获得校级奖学金，荣获优秀毕业生称号。', '校内职务': '担任学生会财务部部长，负责管理学生会资金和预算。', '资格证书': '获得CFA一级证书，通过证券从业资格考试。', '个人作品': '开发了一个基于Python的金融数据分析平台，能够实时抓取和处理金融市场数据，提供可视化分析报告。'}
              单位名称    岗位名称 工作性质     职位分类 工作城市  需求专业   学历要求  \
0  深圳市柏威国际科技物流有限公司  涉外法务助理   全职     法务助理  深圳市    法学  本科及以上   
1  深圳市柏威国际科技物流有限公司  涉外财务助理   全职       财务  深圳市   会计学  本科及以上   
2  深圳市柏威国际科技物流有限公司   空海外客服   全职  物流专员/助理  深圳市  市场营销  本科及以上   

                                         

In [1]:
from typing import List, Dict, Tuple
from langchain_community.embeddings import HuggingFaceEmbeddings  # 导入HuggingFace的文本嵌入模型
from langchain_community.vectorstores import Chroma  # 导入向量数据库
from langchain.text_splitter import CharacterTextSplitter  # 文本分割工具
import os
from dotenv import load_dotenv  # 环境变量加载工具
from chromadb.utils import embedding_functions
import chromadb

In [9]:
# 初始化ResumeJobMatcher
matcher = ResumeJobMatcher(
    os.path.join(ROOT_DIR, "data/SZTU_2025_SPRING_POSITION_TABLE.xlsx"),
    os.path.join(ROOT_DIR, "data/generated_resumes.json")
)

In [13]:

# 存储匹配结果的列表
matching_results = []

# 遍历每一份简历，获取最匹配的10个岗位
for index, resume in enumerate(resumes):
    matched_jobs, resume_text = matcher.match_jobs(index)
    top_10_jobs = matched_jobs[:10]  # 获取最匹配的10个岗位
    
    # 构建结果字典
    result = {
        "唯一标识": resume['基本信息']['唯一标识'],
        "匹配的岗位": top_10_jobs
    }
    
    # 添加到结果列表
    matching_results.append(result)

# 打印匹配结果
for result in matching_results:
    print(f"简历唯一标识: {result['唯一标识']}")
    print("匹配的岗位:")
    for job in result["匹配的岗位"]:
        print(f"- {job['company']} - {job['position']} (匹配度: {job['match_percentage']}%)")
    print("\n")

# 保存匹配结果到JSON文件
output_path = os.path.join(ROOT_DIR, "output/matching_results.json")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(matching_results, f, ensure_ascii=False, indent=4)

print(f"匹配结果已保存到 {output_path}")

UniqueConstraintError: Collection jobs_collection already exists